# Day 5 — Fine-tuning a sentiment classifier

This notebook is a thin caller for the reusable Day 5 pipeline. It fine-tunes `distilbert-base-uncased` for binary SST-2 sentiment classification while keeping the shared outer test rows unavailable until Day 6.

## Resource and evaluation boundary

The full run processes three epochs of SST-2 and may download model/tokenizer files. It can take a long time and substantial memory, especially on CPU. The training cell is disabled by default; enable it only after explicitly approving that cost. During training it prints batch progress every 100 batches and at the end of each epoch. After a successful run it reuses the ignored `fine_tuned_model/` and `fine_tuned_results.txt` artifacts, so the cell skips retraining. Generated data, weights, caches, and result files are ignored by Git.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from transformers_learning import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MODEL_NAME,
    LABEL_COLUMN,
    TEXT_COLUMN,
    SentimentDataset,
    adapt_sst2_split,
    create_fine_tuning_optimizer,
    create_sentiment_dataloaders,
    ensure_sst2_train_data,
    load_sequence_classifier,
    load_tokenizer,
    run_fine_tuning,
    save_fine_tuning_artifacts,
    split_sentiment_row_indices,
)

## Shared data boundaries

The source rows first reproduce Day 4's outer 80/20 split. Only the outer-training rows are split again into train and validation. This gives approximately 64% train, 16% validation, and 20% outer test. Validation monitors the fixed run; it is not the final comparison set. The outer test row IDs are counted below but never passed to the Day 5 datasets or loaders.

In [2]:
sst2_train_path = ensure_sst2_train_data(
    project_root / "data" / "SST-2" / "train.tsv"
)
source_dataframe = pd.read_csv(sst2_train_path, sep="\t")
sentiment_dataframe = adapt_sst2_split(source_dataframe)
row_split = split_sentiment_row_indices(sentiment_dataframe)

train_dataframe = sentiment_dataframe.iloc[row_split.train_indices].reset_index(
    drop=True
)
validation_dataframe = sentiment_dataframe.iloc[
    row_split.validation_indices
].reset_index(drop=True)

print(f"train rows: {len(train_dataframe):,}")
print(f"validation rows: {len(validation_dataframe):,}")
print(f"reserved outer-test rows: {len(row_split.test_indices):,}")

train rows: 43,103
validation rows: 10,776
reserved outer-test rows: 13,470


## Dataset items and batches

`SentimentDataset` tokenizes lazily. One item contains `input_ids: [max_length]`, `attention_mask: [max_length]`, and one scalar label. `DataLoader` stacks items into `[batch, max_length]`, `[batch, max_length]`, and `[batch]`. The label remains separate from tokenization and is used later to calculate classification loss.

In [3]:
model_name = DEFAULT_MODEL_NAME
tokenizer = load_tokenizer(model_name)
train_dataset = SentimentDataset(
    train_dataframe[TEXT_COLUMN].tolist(),
    train_dataframe[LABEL_COLUMN].tolist(),
    tokenizer,
)
validation_dataset = SentimentDataset(
    validation_dataframe[TEXT_COLUMN].tolist(),
    validation_dataframe[LABEL_COLUMN].tolist(),
    tokenizer,
)
dataloaders = create_sentiment_dataloaders(
    train_dataset,
    validation_dataset,
)

sample = train_dataset[0]
validation_batch = next(iter(dataloaders.validation))
print(f"sample text: {train_dataset.texts[0]!r}")
print(f"sample label: {sample['labels'].item()}")
print(f"sample input_ids: {tuple(sample['input_ids'].shape)}")
print(f"sample attention_mask: {tuple(sample['attention_mask'].shape)}")
print(f"batch input_ids: {tuple(validation_batch['input_ids'].shape)}")
print(
    "batch attention_mask: "
    f"{tuple(validation_batch['attention_mask'].shape)}"
)
print(f"batch labels: {tuple(validation_batch['labels'].shape)}")
print(f"configured batch size: {DEFAULT_BATCH_SIZE}")

sample text: 'the best advice '
sample label: 1
sample input_ids: (128,)
sample attention_mask: (128,)
batch input_ids: (16, 128)
batch attention_mask: (16, 128)
batch labels: (16,)
configured batch size: 16


## Training and validation modes

During training, `model.train()` enables training behavior and gradients connect the scalar loss to both the classification head and Transformer parameters. `loss.backward()` calculates those gradients; `optimizer.step()` changes the weights. During validation, `model.eval()` switches evaluation behavior and `torch.no_grad()` prevents graph construction. These are separate responsibilities: `train()` does not update weights by itself, and `eval()` does not disable gradients by itself.

In [ ]:
RUN_FULL_TRAINING = True
model_directory = project_root / "fine_tuned_model"
results_path = project_root / "fine_tuned_results.txt"

if model_directory.is_dir() and results_path.is_file():
    print(f"Found saved fine-tuned model: {model_directory}")
    print("Training skipped; remove these ignored artifacts to train again.")
elif not RUN_FULL_TRAINING:
    print(
        "Full fine-tuning is disabled. Set RUN_FULL_TRAINING = True only "
        "after approving the download, runtime, and memory cost."
    )
else:
    setup = load_sequence_classifier(model_name)
    optimizer = create_fine_tuning_optimizer(setup.model)
    history = run_fine_tuning(
        setup.model,
        dataloaders,
        optimizer,
        setup.device,
        progress_callback=lambda epoch, completed, total: print(
            f"epoch {epoch}/3: {completed:,}/{total:,} batches "
            f"({completed / total * 100:.1f}%)"
        ) if completed % 100 == 0 or completed == total else None,
    )

    for metrics in history:
        print(
            f"epoch={metrics.epoch} "
            f"train_loss={metrics.train_loss:.4f} "
            f"validation_accuracy={metrics.validation_accuracy:.4f} "
            f"validation_macro_f1={metrics.validation_macro_f1:.4f}"
        )

    save_fine_tuning_artifacts(
        setup.model,
        tokenizer,
        history,
        device=setup.device,
        model_directory=model_directory,
        results_path=results_path,
        model_name=model_name,
    )
    print("Saved ignored artifacts after the fixed epoch-3 state.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/3: 100/2,694 batches (3.7%)
epoch 1/3: 200/2,694 batches (7.4%)
epoch 1/3: 300/2,694 batches (11.1%)
epoch 1/3: 400/2,694 batches (14.8%)
epoch 1/3: 500/2,694 batches (18.6%)
epoch 1/3: 600/2,694 batches (22.3%)
epoch 1/3: 700/2,694 batches (26.0%)
epoch 1/3: 800/2,694 batches (29.7%)
epoch 1/3: 900/2,694 batches (33.4%)
epoch 1/3: 1,000/2,694 batches (37.1%)
epoch 1/3: 1,100/2,694 batches (40.8%)
epoch 1/3: 1,200/2,694 batches (44.5%)
epoch 1/3: 1,300/2,694 batches (48.3%)
epoch 1/3: 1,400/2,694 batches (52.0%)
epoch 1/3: 1,500/2,694 batches (55.7%)
epoch 1/3: 1,600/2,694 batches (59.4%)
epoch 1/3: 1,700/2,694 batches (63.1%)
epoch 1/3: 1,800/2,694 batches (66.8%)
epoch 1/3: 1,900/2,694 batches (70.5%)
epoch 1/3: 2,000/2,694 batches (74.2%)
epoch 1/3: 2,100/2,694 batches (78.0%)
epoch 1/3: 2,200/2,694 batches (81.7%)
epoch 1/3: 2,300/2,694 batches (85.4%)
epoch 1/3: 2,400/2,694 batches (89.1%)
epoch 1/3: 2,500/2,694 batches (92.8%)
epoch 1/3: 2,600/2,694 batches (96.5%)
epoch 

## What to remember

- **Day 4 frozen baseline:** Transformer inference used `eval()` and no gradients; fixed `[n_samples, hidden_size]` features were passed to Logistic Regression. The Transformer weights did not change.
- **Day 5 fine-tuning:** token batches `[batch, max_length]` go through a sequence-classification model; gradients update both its classification head and Transformer weights.
- **Train versus validation:** train loss drives parameter updates. Validation accuracy and macro F1 only monitor the fixed three-epoch run.
- **Outer test:** it is excluded from training, validation, early stopping, and hyperparameter choices. It remains reserved for the same-example comparison with the Day 4 baseline in Day 6.
- **Interpretation:** macro F1 gives negative and positive classes equal weight. The recorded Day 5 values are validation metrics, not final held-out test results.